In [ ]:
#imports
import numpy as np
from astropy.visualization import ZScaleInterval, LinearStretch, ImageNormalize
from astropy.io import fits
from photutils.background import Background2D, MedianBackground, SExtractorBackground
from astropy.stats import SigmaClip
import matplotlib.pyplot as plt

In [ ]:
# Open the FITS file
hdul = fits.open('ERO_nonDGL_subregion.fits')

# View the header information (metadata)
print(hdul[0].header)

# Extract the image data into a standard NumPy array
image_data = hdul[0].data
zscale = ZScaleInterval()
vmin, vmax = zscale.get_limits(image_data)

In [ ]:
# Visualize the array
fig, ax = plt.subplots()
ax.imshow(image_data, vmin=vmin, vmax=vmax, origin='upper')
ax.set_title("Z-Scaled FITS Image")
plt.colorbar(ax.imshow(image_data, vmin=vmin, vmax=vmax))
plt.figure(figsize=(50, 50))
plt.show()

In [ ]:
# Close the file when done
# hdul.close()

In [ ]:
# mask stars, 3.0 is aggressive to mask faint stars
sigma_clip = SigmaClip(sigma=2.5, maxiters=10)

# estimate background
bkg_estimator = SExtractorBackground()

# mask the NaN values
mask = ~np.isfinite(image_data)

# 3. Create a 2D model of the background (the "noise" and light gradients)
# box_size defines the grid to evaluate the background over
bkg = Background2D(image_data, box_size=(64, 64), filter_size=(11, 11),
                   sigma_clip=sigma_clip, bkg_estimator=bkg_estimator,
                   mask=mask)

# 4. Subtract the background model from the original raw data
clean_data = image_data - bkg.background

# Visualization: Crucial step to ensure you aren't visually clipping data.
# ZScaleInterval automatically calculates optimal black/white points.
norm = ImageNormalize(clean_data, interval=ZScaleInterval(), stretch=LinearStretch())



# Visualize the Before and After
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(image_data, origin='lower', cmap='gray', vmin=vmin, vmax=vmax) #0, 1000
ax1.set_title('Original Data (with sky glow/noise)')

ax2.imshow(clean_data, origin='lower', cmap='gray', norm=norm)
ax2.set_title('Cleaned Data (background subtracted)')
plt.show()